In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# =====================================================================
# COMPLETE EVALUATION OF test_predictions.csv
#
# Calculates:
# - Normalized Exact Match
# - Token Precision, Recall, F1
# - Fuzzy Match
# - Sentence BLEU
# - Corpus BLEU-4
# - ROUGE-1, ROUGE-2, ROUGE-L
# - METEOR
# - BERTScore Precision, Recall, F1
# - Likely truncated-answer count
#
# Input:
# /kaggle/input/datasets/akra1234/nnsjsjjs/test_predictions.csv
#
# Outputs:
# /kaggle/working/evaluation_per_row.csv
# /kaggle/working/evaluation_summary.csv
# /kaggle/working/truncated_answers.csv
# =====================================================================


# =====================================================================
# 0. INSTALL LIBRARIES
# =====================================================================

!pip install -q \
    pandas \
    numpy \
    torch \
    transformers \
    bert-score \
    sacrebleu \
    rouge-score \
    rapidfuzz \
    nltk


# =====================================================================
# 1. IMPORT LIBRARIES
# =====================================================================

import os
import re
import gc
import unicodedata

from collections import Counter

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
from bert_score import score as calculate_bertscore
from transformers import AutoTokenizer

import warnings
warnings.filterwarnings("ignore")


# =====================================================================
# 2. CONFIGURATION
# =====================================================================

INPUT_PATH = (
    "/kaggle/input/datasets/akra1234/"
    "nnsjsjjs/test_predictions.csv"
)

PER_ROW_OUTPUT_PATH = (
    "/kaggle/working/evaluation_per_row.csv"
)

SUMMARY_OUTPUT_PATH = (
    "/kaggle/working/evaluation_summary.csv"
)

TRUNCATED_OUTPUT_PATH = (
    "/kaggle/working/truncated_answers.csv"
)


# Expected CSV columns
QUESTION_COLUMN = "instruction"
REFERENCE_COLUMN = "output"
PREDICTION_COLUMN = "prediction"


# IMPORTANT:
# Set this to the max_new_tokens used when generating this CSV.
#
# Original pipeline:
#     MAX_NEW_TOKENS = 256
#
# Improved pipeline:
#     MAX_NEW_TOKENS = 512 or 1024
#
# For your original test_predictions.csv, 256 is probably correct.
GENERATION_MAX_NEW_TOKENS = 256


# Qwen tokenizer is used only to estimate generated token counts.
LOCAL_TOKENIZER_PATH = (
    "/kaggle/working/"
    "qwen25_7b_training_checkpoints/checkpoint-298"
)

FALLBACK_TOKENIZER_NAME = (
    "Qwen/Qwen2.5-7B-Instruct"
)


# Multilingual BERTScore model
BERTSCORE_MODEL = "bert-base-multilingual-cased"

BERTSCORE_BATCH_SIZE = 8


# =====================================================================
# 3. VALIDATE AND LOAD CSV
# =====================================================================

print("=" * 80)
print("LOADING PREDICTION FILE")
print("=" * 80)

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Prediction file was not found:\n{INPUT_PATH}"
    )

test_df = pd.read_csv(
    INPUT_PATH,
    encoding="utf-8",
)

print("Rows:", len(test_df))
print("Columns:", test_df.columns.tolist())


required_columns = [
    REFERENCE_COLUMN,
    PREDICTION_COLUMN,
]

missing_columns = [
    column
    for column in required_columns
    if column not in test_df.columns
]

if missing_columns:
    raise ValueError(
        "The following required columns are missing:\n"
        f"{missing_columns}\n\n"
        f"Available columns:\n{test_df.columns.tolist()}"
    )


# Question column is useful but not compulsory for metric calculation.
if QUESTION_COLUMN not in test_df.columns:
    print(
        f"\nWarning: '{QUESTION_COLUMN}' was not found. "
        "Metric calculation will continue."
    )


test_df[REFERENCE_COLUMN] = (
    test_df[REFERENCE_COLUMN]
    .fillna("")
    .astype(str)
    .str.strip()
)

test_df[PREDICTION_COLUMN] = (
    test_df[PREDICTION_COLUMN]
    .fillna("")
    .astype(str)
    .str.strip()
)


print("\nSample rows:")

display_columns = [
    column
    for column in [
        QUESTION_COLUMN,
        REFERENCE_COLUMN,
        PREDICTION_COLUMN,
    ]
    if column in test_df.columns
]

display(
    test_df[display_columns].head(10)
)


# =====================================================================
# 4. NORMALIZE BANGLA AND MIXED-LANGUAGE TEXT
# =====================================================================

def normalize_text(text):
    """
    Normalize Bangla and mixed-language text for lexical evaluation.

    Steps:
    1. Unicode NFKC normalization
    2. Lowercase English characters
    3. Replace punctuation and control characters with spaces
    4. Normalize repeated whitespace
    """

    if text is None:
        return ""

    if isinstance(text, float) and np.isnan(text):
        return ""

    text = unicodedata.normalize(
        "NFKC",
        str(text),
    )

    text = text.lower()

    normalized_characters = []

    for character in text:

        category = unicodedata.category(
            character
        )

        if (
            category.startswith("P")
            or category.startswith("C")
        ):
            normalized_characters.append(" ")

        else:
            normalized_characters.append(
                character
            )

    text = "".join(
        normalized_characters
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


test_df["normalized_reference"] = (
    test_df[REFERENCE_COLUMN]
    .apply(normalize_text)
)

test_df["normalized_prediction"] = (
    test_df[PREDICTION_COLUMN]
    .apply(normalize_text)
)


normalized_references = (
    test_df["normalized_reference"]
    .tolist()
)

normalized_predictions = (
    test_df["normalized_prediction"]
    .tolist()
)


# =====================================================================
# 5. NORMALIZED EXACT MATCH
# =====================================================================

test_df["normalized_exact_match"] = [
    float(reference == prediction)
    for reference, prediction in zip(
        normalized_references,
        normalized_predictions,
    )
]


# =====================================================================
# 6. TOKEN PRECISION, RECALL AND F1
# =====================================================================

def whitespace_tokenize(text):
    if not text:
        return []

    return text.split()


def calculate_token_metrics(
    reference,
    prediction,
):
    """
    Calculate bag-of-token precision, recall and F1.
    Repeated token frequency is considered.
    """

    reference_tokens = whitespace_tokenize(
        reference
    )

    prediction_tokens = whitespace_tokenize(
        prediction
    )

    if (
        len(reference_tokens) == 0
        and len(prediction_tokens) == 0
    ):
        return 1.0, 1.0, 1.0

    if (
        len(reference_tokens) == 0
        or len(prediction_tokens) == 0
    ):
        return 0.0, 0.0, 0.0


    reference_counts = Counter(
        reference_tokens
    )

    prediction_counts = Counter(
        prediction_tokens
    )

    common_counts = (
        reference_counts
        & prediction_counts
    )

    shared_token_count = sum(
        common_counts.values()
    )

    precision = (
        shared_token_count
        / len(prediction_tokens)
    )

    recall = (
        shared_token_count
        / len(reference_tokens)
    )

    if precision + recall == 0:
        f1 = 0.0

    else:
        f1 = (
            2
            * precision
            * recall
            / (precision + recall)
        )

    return precision, recall, f1


token_results = [
    calculate_token_metrics(
        reference,
        prediction,
    )
    for reference, prediction in zip(
        normalized_references,
        normalized_predictions,
    )
]


test_df["token_precision"] = [
    result[0]
    for result in token_results
]

test_df["token_recall"] = [
    result[1]
    for result in token_results
]

test_df["token_f1"] = [
    result[2]
    for result in token_results
]


# =====================================================================
# 7. FUZZY MATCH
# =====================================================================

def calculate_fuzzy_match(
    reference,
    prediction,
):
    if not reference and not prediction:
        return 1.0

    if not reference or not prediction:
        return 0.0

    return (
        fuzz.token_set_ratio(
            reference,
            prediction,
        )
        / 100.0
    )


test_df["fuzzy_match"] = [
    calculate_fuzzy_match(
        reference,
        prediction,
    )
    for reference, prediction in zip(
        normalized_references,
        normalized_predictions,
    )
]


# =====================================================================
# 8. ROUGE-1, ROUGE-2 AND ROUGE-L
# =====================================================================

class BanglaWhitespaceTokenizer:
    def tokenize(self, text):
        return whitespace_tokenize(text)


rouge_calculator = rouge_scorer.RougeScorer(
    rouge_types=[
        "rouge1",
        "rouge2",
        "rougeL",
    ],
    use_stemmer=False,
    tokenizer=BanglaWhitespaceTokenizer(),
)


rouge_results = []

for reference, prediction in tqdm(
    zip(
        normalized_references,
        normalized_predictions,
    ),
    total=len(test_df),
    desc="Calculating ROUGE",
):

    if not reference and not prediction:

        row_result = {
            "rouge1": (1.0, 1.0, 1.0),
            "rouge2": (1.0, 1.0, 1.0),
            "rougeL": (1.0, 1.0, 1.0),
        }

    elif not reference or not prediction:

        row_result = {
            "rouge1": (0.0, 0.0, 0.0),
            "rouge2": (0.0, 0.0, 0.0),
            "rougeL": (0.0, 0.0, 0.0),
        }

    else:

        scores = rouge_calculator.score(
            reference,
            prediction,
        )

        row_result = {
            rouge_name: (
                scores[rouge_name].precision,
                scores[rouge_name].recall,
                scores[rouge_name].fmeasure,
            )
            for rouge_name in [
                "rouge1",
                "rouge2",
                "rougeL",
            ]
        }

    rouge_results.append(row_result)


for rouge_name in [
    "rouge1",
    "rouge2",
    "rougeL",
]:

    test_df[
        f"{rouge_name}_precision"
    ] = [
        result[rouge_name][0]
        for result in rouge_results
    ]

    test_df[
        f"{rouge_name}_recall"
    ] = [
        result[rouge_name][1]
        for result in rouge_results
    ]

    test_df[
        f"{rouge_name}_f1"
    ] = [
        result[rouge_name][2]
        for result in rouge_results
    ]


# =====================================================================
# 9. METEOR WITHOUT ENGLISH WORDNET
# =====================================================================

class EmptyWordNet:
    """
    Disable English WordNet synonym matching because WordNet
    synonym matching is unsuitable for Bangla evaluation.
    """

    @staticmethod
    def synsets(*args, **kwargs):
        return []


def calculate_meteor(
    reference,
    prediction,
):
    reference_tokens = whitespace_tokenize(
        reference
    )

    prediction_tokens = whitespace_tokenize(
        prediction
    )

    if (
        not reference_tokens
        and not prediction_tokens
    ):
        return 1.0

    if (
        not reference_tokens
        or not prediction_tokens
    ):
        return 0.0

    return meteor_score(
        references=[reference_tokens],
        hypothesis=prediction_tokens,
        wordnet=EmptyWordNet(),
    )


test_df["meteor"] = [
    calculate_meteor(
        reference,
        prediction,
    )
    for reference, prediction in tqdm(
        zip(
            normalized_references,
            normalized_predictions,
        ),
        total=len(test_df),
        desc="Calculating METEOR",
    )
]


# =====================================================================
# 10. SENTENCE BLEU AND CORPUS BLEU-4
# =====================================================================

bleu_calculator = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True,
)


sentence_bleu_scores = []

for reference, prediction in tqdm(
    zip(
        normalized_references,
        normalized_predictions,
    ),
    total=len(test_df),
    desc="Calculating BLEU",
):

    if not reference and not prediction:
        sentence_bleu = 1.0

    elif not reference or not prediction:
        sentence_bleu = 0.0

    else:
        result = bleu_calculator.sentence_score(
            hypothesis=prediction,
            references=[reference],
        )

        sentence_bleu = (
            result.score / 100.0
        )

    sentence_bleu_scores.append(
        sentence_bleu
    )


test_df["sentence_bleu"] = (
    sentence_bleu_scores
)


corpus_bleu_result = (
    bleu_calculator.corpus_score(
        hypotheses=normalized_predictions,
        references=[normalized_references],
    )
)

corpus_bleu_score = (
    corpus_bleu_result.score / 100.0
)


# =====================================================================
# 11. BERTSCORE
# =====================================================================

print("\n" + "=" * 80)
print("CALCULATING MULTILINGUAL BERTSCORE")
print("=" * 80)


original_references = (
    test_df[REFERENCE_COLUMN]
    .fillna("")
    .astype(str)
    .tolist()
)

original_predictions = (
    test_df[PREDICTION_COLUMN]
    .fillna("")
    .astype(str)
    .tolist()
)


# BERTScore can behave inconsistently with empty strings.
bertscore_references = [
    text if text.strip() else "[EMPTY]"
    for text in original_references
]

bertscore_predictions = [
    text if text.strip() else "[EMPTY]"
    for text in original_predictions
]


bertscore_device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("BERTScore device:", bertscore_device)


try:

    bert_precision, bert_recall, bert_f1 = (
        calculate_bertscore(
            cands=bertscore_predictions,
            refs=bertscore_references,
            model_type=BERTSCORE_MODEL,
            device=bertscore_device,
            batch_size=BERTSCORE_BATCH_SIZE,
            verbose=True,
            rescale_with_baseline=False,
        )
    )


    test_df["bertscore_precision"] = (
        bert_precision
        .detach()
        .cpu()
        .numpy()
    )

    test_df["bertscore_recall"] = (
        bert_recall
        .detach()
        .cpu()
        .numpy()
    )

    test_df["bertscore_f1"] = (
        bert_f1
        .detach()
        .cpu()
        .numpy()
    )


    # Correct empty-reference/prediction cases.
    for row_index, (
        normalized_reference,
        normalized_prediction,
    ) in enumerate(
        zip(
            normalized_references,
            normalized_predictions,
        )
    ):

        if (
            not normalized_reference
            and not normalized_prediction
        ):

            test_df.at[
                row_index,
                "bertscore_precision",
            ] = 1.0

            test_df.at[
                row_index,
                "bertscore_recall",
            ] = 1.0

            test_df.at[
                row_index,
                "bertscore_f1",
            ] = 1.0

        elif (
            not normalized_reference
            or not normalized_prediction
        ):

            test_df.at[
                row_index,
                "bertscore_precision",
            ] = 0.0

            test_df.at[
                row_index,
                "bertscore_recall",
            ] = 0.0

            test_df.at[
                row_index,
                "bertscore_f1",
            ] = 0.0


except RuntimeError as error:

    if "out of memory" in str(error).lower():

        print(
            "\nCUDA out-of-memory occurred during BERTScore."
        )

        print(
            "Change BERTSCORE_BATCH_SIZE from 8 to 4 or 2 "
            "and rerun the cell."
        )

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    raise


# =====================================================================
# 12. LOAD QWEN TOKENIZER FOR TRUNCATION CHECKING
# =====================================================================

print("\n" + "=" * 80)
print("CHECKING FOR TRUNCATED ANSWERS")
print("=" * 80)


if os.path.exists(LOCAL_TOKENIZER_PATH):

    tokenizer_source = LOCAL_TOKENIZER_PATH

else:

    tokenizer_source = FALLBACK_TOKENIZER_NAME


print("Tokenizer source:", tokenizer_source)
print(
    "Expected generation limit:",
    GENERATION_MAX_NEW_TOKENS,
)


generation_tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_source,
    trust_remote_code=True,
)


def count_model_tokens(text):
    """
    Count Qwen tokenizer tokens without adding chat-template tokens.
    """

    if not str(text).strip():
        return 0

    return len(
        generation_tokenizer.encode(
            str(text),
            add_special_tokens=False,
        )
    )


test_df["prediction_model_token_count"] = [
    count_model_tokens(prediction)
    for prediction in tqdm(
        test_df[PREDICTION_COLUMN].tolist(),
        desc="Counting prediction tokens",
    )
]


# =====================================================================
# 13. TRUNCATION DETECTION
# =====================================================================

def convert_to_boolean(value):
    """
    Convert CSV boolean-like values into real booleans.
    """

    if isinstance(value, bool):
        return value

    if pd.isna(value):
        return False

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
    }


truncation_method = None


# ---------------------------------------------------------
# Method 1: Use exact diagnostic column if available
# ---------------------------------------------------------

if "generation_still_truncated" in test_df.columns:

    test_df["likely_truncated"] = (
        test_df["generation_still_truncated"]
        .apply(convert_to_boolean)
    )

    truncation_method = (
        "generation_still_truncated diagnostic column"
    )


# ---------------------------------------------------------
# Method 2: Use hit_token_limit if available
# ---------------------------------------------------------

elif "hit_token_limit" in test_df.columns:

    test_df["likely_truncated"] = (
        test_df["hit_token_limit"]
        .apply(convert_to_boolean)
    )

    truncation_method = (
        "hit_token_limit diagnostic column"
    )


# ---------------------------------------------------------
# Method 3: Use generation count + EOS columns
# ---------------------------------------------------------

elif (
    "generation_token_count" in test_df.columns
    and "generation_finished_with_eos" in test_df.columns
):

    generated_counts = pd.to_numeric(
        test_df["generation_token_count"],
        errors="coerce",
    ).fillna(0)

    finished_with_eos = (
        test_df["generation_finished_with_eos"]
        .apply(convert_to_boolean)
    )

    test_df["likely_truncated"] = (
        (
            generated_counts
            >= GENERATION_MAX_NEW_TOKENS
        )
        & (~finished_with_eos)
    )

    truncation_method = (
        "generation token count and EOS diagnostic"
    )


# ---------------------------------------------------------
# Method 4: Token-count approximation
# ---------------------------------------------------------

else:

    # Re-encoding decoded text may produce a count slightly below
    # the original generated-token count because EOS and other
    # special tokens are not present in the decoded prediction.
    truncation_threshold = max(
        1,
        GENERATION_MAX_NEW_TOKENS - 4,
    )

    test_df["likely_truncated"] = (
        test_df["prediction_model_token_count"]
        >= truncation_threshold
    )

    truncation_method = (
        "Qwen tokenizer count approximation"
    )


# Extra textual diagnostic: does the answer appear to end cleanly?
terminal_pattern = re.compile(
    r"[।.!?！？]\s*$"
)

test_df["ends_with_terminal_punctuation"] = (
    test_df[PREDICTION_COLUMN]
    .apply(
        lambda text: bool(
            terminal_pattern.search(
                str(text).strip()
            )
        )
    )
)


test_df["likely_truncated_without_clean_ending"] = (
    test_df["likely_truncated"]
    & (
        ~test_df[
            "ends_with_terminal_punctuation"
        ]
    )
)


truncated_count = int(
    test_df["likely_truncated"].sum()
)

stronger_truncated_count = int(
    test_df[
        "likely_truncated_without_clean_ending"
    ].sum()
)


print("Truncation detection method:", truncation_method)

print(
    "Likely truncated answers:",
    truncated_count,
)

print(
    "Likely truncated and no final punctuation:",
    stronger_truncated_count,
)

print(
    "Likely truncated percentage:",
    round(
        truncated_count
        / max(len(test_df), 1)
        * 100,
        2,
    ),
    "%",
)


# =====================================================================
# 14. ADD BASIC DIAGNOSTIC COLUMNS
# =====================================================================

test_df["reference_whitespace_token_count"] = (
    test_df["normalized_reference"]
    .apply(
        lambda text: len(
            whitespace_tokenize(text)
        )
    )
)

test_df["prediction_whitespace_token_count"] = (
    test_df["normalized_prediction"]
    .apply(
        lambda text: len(
            whitespace_tokenize(text)
        )
    )
)

test_df["empty_prediction"] = (
    test_df["normalized_prediction"] == ""
)


# =====================================================================
# 15. CREATE PAPER-READY SUMMARY
# =====================================================================

def percentage_mean(column_name):
    return (
        test_df[column_name].mean()
        * 100.0
    )


summary_df = pd.DataFrame(
    {
        "Metric": [
            "Normalized Exact Match",
            "Token Precision",
            "Token Recall",
            "Token F1",
            "Fuzzy Match",
            "Sentence BLEU",
            "Corpus BLEU-4",
            "ROUGE-1 Precision",
            "ROUGE-1 Recall",
            "ROUGE-1 F1",
            "ROUGE-2 Precision",
            "ROUGE-2 Recall",
            "ROUGE-2 F1",
            "ROUGE-L Precision",
            "ROUGE-L Recall",
            "ROUGE-L F1",
            "METEOR",
            "BERTScore Precision",
            "BERTScore Recall",
            "BERTScore F1",
        ],

        "Score (%)": [
            percentage_mean(
                "normalized_exact_match"
            ),

            percentage_mean(
                "token_precision"
            ),

            percentage_mean(
                "token_recall"
            ),

            percentage_mean(
                "token_f1"
            ),

            percentage_mean(
                "fuzzy_match"
            ),

            percentage_mean(
                "sentence_bleu"
            ),

            corpus_bleu_score * 100.0,

            percentage_mean(
                "rouge1_precision"
            ),

            percentage_mean(
                "rouge1_recall"
            ),

            percentage_mean(
                "rouge1_f1"
            ),

            percentage_mean(
                "rouge2_precision"
            ),

            percentage_mean(
                "rouge2_recall"
            ),

            percentage_mean(
                "rouge2_f1"
            ),

            percentage_mean(
                "rougeL_precision"
            ),

            percentage_mean(
                "rougeL_recall"
            ),

            percentage_mean(
                "rougeL_f1"
            ),

            percentage_mean(
                "meteor"
            ),

            percentage_mean(
                "bertscore_precision"
            ),

            percentage_mean(
                "bertscore_recall"
            ),

            percentage_mean(
                "bertscore_f1"
            ),
        ],
    }
)


summary_df["Score (%)"] = (
    summary_df["Score (%)"]
    .round(2)
)


# Add diagnostic information separately.
diagnostic_summary = pd.DataFrame(
    {
        "Diagnostic": [
            "Total evaluated answers",
            "Empty predictions",
            "Likely truncated answers",
            "Likely truncated answers (%)",
            "Truncated without final punctuation",
            "Generation token limit used",
        ],

        "Value": [
            len(test_df),

            int(
                test_df[
                    "empty_prediction"
                ].sum()
            ),

            truncated_count,

            round(
                truncated_count
                / max(len(test_df), 1)
                * 100,
                2,
            ),

            stronger_truncated_count,

            GENERATION_MAX_NEW_TOKENS,
        ],
    }
)


# =====================================================================
# 16. SAVE RESULTS
# =====================================================================

test_df.to_csv(
    PER_ROW_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


summary_df.to_csv(
    SUMMARY_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


truncated_df = test_df[
    test_df["likely_truncated"]
].copy()


truncated_df.to_csv(
    TRUNCATED_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)


print("\n" + "=" * 80)
print("FINAL EVALUATION METRICS")
print("=" * 80)

display(summary_df)


print("\n" + "=" * 80)
print("TRUNCATION AND DATA DIAGNOSTICS")
print("=" * 80)

display(diagnostic_summary)


print("\n" + "=" * 80)
print("LIKELY TRUNCATED ANSWERS")
print("=" * 80)


truncated_display_columns = [
    column
    for column in [
        QUESTION_COLUMN,
        REFERENCE_COLUMN,
        PREDICTION_COLUMN,
        "prediction_model_token_count",
        "likely_truncated",
        "ends_with_terminal_punctuation",
    ]
    if column in truncated_df.columns
]


display(
    truncated_df[
        truncated_display_columns
    ].head(50)
)


print("\n" + "=" * 80)
print("SAVED OUTPUTS")
print("=" * 80)

print(
    "Per-row metrics:\n",
    PER_ROW_OUTPUT_PATH,
)

print(
    "\nEvaluation summary:\n",
    SUMMARY_OUTPUT_PATH,
)

print(
    "\nLikely truncated answers:\n",
    TRUNCATED_OUTPUT_PATH,
)